# Modulo 7 · Lezione 8 — Clustering

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | Apprendimento non supervisionato: cos'è e quando usarlo |
| 2 | K-Means: algoritmo passo dopo passo |
| 3 | Elbow method: scegliere il numero di cluster |
| 4 | Silhouette score: qualità del clustering |
| 5 | Caso reale: segmentazione clienti |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.datasets import make_blobs

np.random.seed(42)
print("Librerie pronte")

---
## 1. Apprendimento non supervisionato

```
Supervisionato:     i dati hanno etichette (y)
                 obiettivo: predire y per dati nuovi

Non supervisionato:  dati SENZA etichette
                 obiettivo: scoprire STRUTTURA nei dati

Casi d'uso del clustering:
  - Segmentazione clienti (marketing)
  - Raggruppamento di documenti / notizie
  - Rilevamento di anomalie
  - Compressione di immagini
  - Bioinformatica: gruppi di geni con comportamento simile

Algoritmi più comuni:
  K-Means           → gruppi sferici, dimensione simile
  DBSCAN            → gruppi di forma arbitraria, outlier espliciti
  Gerarchico        → dendrogramma, non richiede K
  Gaussian Mixture  → gruppi sovrapposti con probabilità

Oggi: K-Means
```

---
## 2. K-Means: algoritmo

```
Parametro richiesto: K (numero di cluster)

Algoritmo:
  1. Inizializzare K centroidi a caso
  2. Assegnare ogni punto al centroide più vicino (distanza euclidea)
  3. Ricalcolare i centroidi come media dei punti assegnati
  4. Ripetere 2-3 fino a convergenza (i centroidi non si spostano più)

Criterio di qualità: inerzia (dentro il cluster)
  Inerzia = Σ dist²(punto, centroide_assegnato)
  Inerzia minore = cluster più compatti

Limitazioni:
  - Richiede di specificare K in anticipo
  - Sensibile agli outlier
  - Presuppone cluster sferici e di dimensione simile
  - Il risultato dipende dall'inizializzazione (usa k-means++ di default)
```

In [ ]:
# Visualizzare il processo di K-Means passo dopo passo
np.random.seed(42)
X_demo, y_true = make_blobs(n_samples=120, centers=3, cluster_std=1.2, random_state=42)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
iteraciones = [1, 2, 4, 10]
colores_cl  = ['tomato', 'steelblue', 'seagreen']

for ax, n_iter in zip(axes, iteraciones):
    km = KMeans(n_clusters=3, max_iter=n_iter, n_init=1,
                init='random', random_state=0)
    labels = km.fit_predict(X_demo)
    for cl in range(3):
        mask = labels == cl
        ax.scatter(X_demo[mask, 0], X_demo[mask, 1],
                   c=colores_cl[cl], s=25, alpha=0.7)
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               c='black', s=200, marker='X', zorder=10)
    ax.set_title(f'Iterazione {n_iter}\nInerzia={km.inertia_:.1f}')

plt.suptitle('K-Means: convergenza iterativa', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Uso base di KMeans in sklearn
km = KMeans(
    n_clusters=3,    # numero di cluster K
    n_init=10,       # riavviare 10 volte con centroidi diversi, tenere il migliore
    max_iter=300,    # massimo numero di iterazioni
    random_state=42  # riproducibilità
)
labels = km.fit_predict(X_demo)  # fit + assegnare le etichette in un solo passo

print(f"Etichette uniche: {np.unique(labels)}")
print(f"Dimensione dei cluster: {np.bincount(labels)}")
print(f"Inerzia: {km.inertia_:.2f}")
print(f"Centroidi:\n{km.cluster_centers_.round(2)}")
print(f"Iterazioni fino a convergenza: {km.n_iter_}")

# Visualizzare il risultato finale
fig, ax = plt.subplots(figsize=(6, 4))
for cl in range(3):
    mask = labels == cl
    ax.scatter(X_demo[mask, 0], X_demo[mask, 1],
               label=f'Cluster {cl}', s=30, alpha=0.8)
ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
           c='black', s=300, marker='X', label='Centroidi', zorder=10)
ax.set_title('K-Means finale (K=3)')
ax.legend(); plt.tight_layout(); plt.show()

---
## 3. Elbow Method: scegliere K

In [ ]:
# Il trucco: l'inerzia scende sempre aumentando i cluster
# Cercare il "gomito" dove il miglioramento inizia a essere marginale

np.random.seed(42)
X_cl, _ = make_blobs(n_samples=300, centers=4, cluster_std=1.5, random_state=42)

ks      = range(1, 12)
inercias = []
for k in ks:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42)
    km_k.fit(X_cl)
    inercias.append(km_k.inertia_)

# Calcolare la derivata seconda per individuare il gomito automaticamente
deltas  = np.diff(inercias)
deltas2 = np.diff(deltas)
k_codo  = int(np.argmax(np.abs(deltas2)) + 2)  # +2 per i due diff

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(ks, inercias, 'bo-', ms=7, lw=2)
axes[0].axvline(k_codo, color='red', ls='--', lw=2, label=f'Gomito a K={k_codo}')
axes[0].set_xlabel('K (numero di cluster)')
axes[0].set_ylabel('Inerzia')
axes[0].set_title('Elbow Method')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Riduzione relativa dell'inerzia
reduccion = [-deltas[i] / inercias[i] * 100 for i in range(len(deltas))]
axes[1].bar(range(2, len(ks)+1), reduccion, color='steelblue')
axes[1].axvline(k_codo, color='red', ls='--', lw=2, label=f'K ottimale={k_codo}')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Riduzione dell\'inerzia (%)')
axes[1].set_title('Riduzione relativa aggiungendo ogni cluster')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f"K ottimale suggerito dall'elbow: {k_codo} (i dati reali avevano 4 cluster)")

---
## 4. Silhouette Score

```
Silhouette di un punto i:
  a(i) = distanza media dai punti dello STESSO cluster
  b(i) = distanza media dal cluster PIÙ VICINO (diverso)
  s(i) = (b(i) - a(i)) / max(a(i), b(i))

Interpretazione:
  s(i) vicino a +1 → punto ben assegnato
  s(i) vicino a 0  → punto al confine tra cluster
  s(i) < 0         → punto probabilmente mal assegnato

Silhouette Score globale = media di tutti gli s(i)
  Intervallo: [-1, 1]. Cercare il K che massimizza questo valore.
```

In [ ]:
# Silhouette score per diversi K
silhouettes = []
for k in range(2, 10):
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels_k = km_k.fit_predict(X_cl)
    sc = silhouette_score(X_cl, labels_k)
    silhouettes.append(sc)
    print(f"K={k}: silhouette={sc:.4f}, inerzia={km_k.inertia_:.1f}")

k_opt = range(2, 10)[np.argmax(silhouettes)]
print(f"\nK ottimale per silhouette: {k_opt}")

# Confrontare elbow vs silhouette
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(2, 10), silhouettes, 'rs-', ms=7, lw=2)
ax.axvline(k_opt, color='red', ls='--', lw=2, label=f'K ottimale={k_opt}')
ax.set_xlabel('K'); ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score vs K')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Grafico di silhouette per cluster
def plot_silhouette(X, k):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    sc_global = silhouette_score(X, labels)
    sc_muestra = silhouette_samples(X, labels)

    fig, ax = plt.subplots(figsize=(7, 5))
    y_lower = 10
    colores = cm.Set2(np.linspace(0, 0.8, k))
    for cl in range(k):
        sc_cl = np.sort(sc_muestra[labels == cl])
        size_cl = len(sc_cl)
        y_upper = y_lower + size_cl
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, sc_cl,
                         facecolor=colores[cl], alpha=0.8)
        ax.text(-0.05, y_lower + size_cl/2, str(cl), fontsize=9)
        y_lower = y_upper + 5

    ax.axvline(sc_global, color='red', ls='--', lw=2)
    ax.set_title(f'Silhouette Plot (K={k}, score globale={sc_global:.3f})')
    ax.set_xlabel('Silhouette individuale')
    ax.set_ylabel('Cluster')
    ax.set_yticks([]); ax.set_xlim([-0.2, 1.0])
    plt.tight_layout(); plt.show()

# Vedere K=3 e K=4 (quello corretto)
for k in [3, 4]:
    plot_silhouette(X_cl, k)

---
## 5. Caso reale: segmentazione clienti

In [ ]:
# Dataset sintetico di clienti di un e-commerce
np.random.seed(42)
n = 400
# Profilo A: clienti frequenti e ad alto valore
A_gasto  = np.random.normal(500, 80, n//4)
A_visitas = np.random.normal(20, 3, n//4)
A_edad   = np.random.normal(35, 5, n//4)
# Profilo B: clienti occasionali e sensibili al prezzo
B_gasto  = np.random.normal(100, 30, n//4)
B_visitas = np.random.normal(4, 1.5, n//4)
B_edad   = np.random.normal(25, 4, n//4)
# Profilo C: clienti maturi e stabili
C_gasto  = np.random.normal(300, 50, n//4)
C_visitas = np.random.normal(10, 2, n//4)
C_edad   = np.random.normal(55, 8, n//4)
# Profilo D: clienti aziendali
D_gasto  = np.random.normal(1200, 200, n//4)
D_visitas = np.random.normal(8, 2, n//4)
D_edad   = np.random.normal(45, 6, n//4)

df_clientes = pd.DataFrame({
    'gasto_mensual': np.concatenate([A_gasto, B_gasto, C_gasto, D_gasto]),
    'visitas_mes':   np.concatenate([A_visitas, B_visitas, C_visitas, D_visitas]),
    'edad':          np.concatenate([A_edad, B_edad, C_edad, D_edad]),
})
df_clientes = df_clientes.clip(lower=0)  # senza negativi

print("Dataset di clienti:")
print(df_clientes.describe().round(1))

In [ ]:
# 1. Scalare i dati (SEMPRE prima del clustering)
scaler = StandardScaler()
X_cl_sc = scaler.fit_transform(df_clientes)

# 2. Elbow + Silhouette per scegliere K
ks_rango = range(2, 9)
inercias_cl, siluetas_cl = [], []
for k in ks_rango:
    km_k = KMeans(k, n_init=15, random_state=42)
    lab  = km_k.fit_predict(X_cl_sc)
    inercias_cl.append(km_k.inertia_)
    siluetas_cl.append(silhouette_score(X_cl_sc, lab))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ks_rango, inercias_cl, 'bo-', ms=7, lw=2)
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inerzia')
axes[0].set_title('Elbow Method'); axes[0].grid(True, alpha=0.3)

k_opt_sil = list(ks_rango)[np.argmax(siluetas_cl)]
axes[1].plot(ks_rango, siluetas_cl, 'rs-', ms=7, lw=2)
axes[1].axvline(k_opt_sil, color='red', ls='--', label=f'K ottimale={k_opt_sil}')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette')
axes[1].set_title('Silhouette Score'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Clienti: scegliere il numero di segmenti', fontsize=11)
plt.tight_layout(); plt.show()
print(f"K ottimale per silhouette: {k_opt_sil}")

In [ ]:
# 3. Allenare KMeans con K ottimale
K_FINAL = k_opt_sil
km_final = KMeans(K_FINAL, n_init=20, random_state=42)
df_clientes['segmento'] = km_final.fit_predict(X_cl_sc)

# 4. Caratterizzare ogni segmento
perfil = df_clientes.groupby('segmento').agg(
    n_clientes     = ('gasto_mensual', 'count'),
    gasto_medio    = ('gasto_mensual', 'mean'),
    visitas_media  = ('visitas_mes', 'mean'),
    edad_media     = ('edad', 'mean'),
).round(1)
print("Profilo dei segmenti:")
print(perfil)

# 5. Assegnare nomi commerciali
orden_gasto = perfil['gasto_medio'].rank(ascending=False)
nombres = {}
for seg in perfil.index:
    rk = int(orden_gasto[seg])
    if rk == 1:   nombres[seg] = 'Premium'
    elif rk == 2: nombres[seg] = 'Frequente'
    elif rk == 3: nombres[seg] = 'Stabile'
    else:         nombres[seg] = 'Occasionale'

df_clientes['nombre_segmento'] = df_clientes['segmento'].map(nombres)
print(f"\nEtichette assegnate: {nombres}")

# 6. Visualizzare
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colores_seg = ['#e74c3c','#3498db','#2ecc71','#9b59b6']
for seg in range(K_FINAL):
    mask = df_clientes['segmento'] == seg
    axes[0].scatter(df_clientes.loc[mask,'gasto_mensual'],
                    df_clientes.loc[mask,'visitas_mes'],
                    c=colores_seg[seg], s=25, alpha=0.7,
                    label=nombres[seg])
axes[0].set_xlabel('Spesa mensile (€)')
axes[0].set_ylabel('Visite al mese')
axes[0].set_title('Segmentazione clienti')
axes[0].legend()

n_segs = df_clientes.groupby('nombre_segmento').size()
axes[1].pie(n_segs, labels=n_segs.index, autopct='%1.1f%%',
            colors=colores_seg[:len(n_segs)],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Distribuzione dei segmenti')
plt.tight_layout(); plt.show()

---
## Pratica

### Esercizio 1 — Clustering su dati reali

**Consegna:** Usa il dataset iris senza le etichette:

```python
from sklearn.datasets import load_iris
iris = load_iris()
X = iris.data  # 4 feature: sepalo e petalo
y_real = iris.target  # NON usare per allenare KMeans
```

1. Applica Elbow Method e Silhouette Score per K da 2 a 8
2. Allena KMeans con il K ottimale
3. Confronta i cluster trovati con le etichette reali (confusion matrix)
4. Quale percentuale di campioni ha classificato correttamente (circa)?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, confusion_matrix

iris = load_iris()
X, y_real = iris.data, iris.target

scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

# Elbow + Silhouette
ks = range(2, 9)
inercias, siluetas = [], []
for k in ks:
    km = KMeans(k, n_init=15, random_state=42)
    lab = km.fit_predict(X_sc)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_sc, lab))

k_opt = list(ks)[np.argmax(siluetas)]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ks, inercias, 'bo-', ms=7); axes[0].set_title('Elbow'); axes[0].set_xlabel('K')
axes[1].plot(ks, siluetas, 'rs-', ms=7); axes[1].set_title('Silhouette'); axes[1].set_xlabel('K')
axes[1].axvline(k_opt, color='red', ls='--', label=f'K={k_opt}')
axes[1].legend(); plt.tight_layout(); plt.show()
print(f"K ottimale: {k_opt}")

# KMeans finale
km_final = KMeans(k_opt, n_init=20, random_state=42)
labels = km_final.fit_predict(X_sc)

# Confrontare con le etichette reali
# Allineare le labels (possono essere in ordine diverso)
from scipy.optimize import linear_sum_assignment
cm = confusion_matrix(y_real, labels)
row_ind, col_ind = linear_sum_assignment(-cm)
acc_aprox = cm[row_ind, col_ind].sum() / len(y_real)
print(f"\nMatrice di confusione (cluster vs classe reale):")
print(cm)
print(f"\nAccuracy approssimata (allineando i cluster): {acc_aprox:.2%}")
print("Iris-versicolor e iris-virginica si sovrappongono: il clustering lo riflette.")
```
</details>

---
### Esercizio 2 — Effetto dello scaling

**Consegna:** Genera dati con feature su scale molto diverse:

```python
np.random.seed(42)
X_sin_escala = np.column_stack([
    np.random.randn(200) * 1000,   # feature A: intervallo 0–3000
    np.random.randn(200)           # feature B: intervallo -3 a 3
])
```

1. Applica KMeans (K=3) SENZA scaling e visualizza i cluster
2. Applica KMeans (K=3) CON StandardScaler e visualizza
3. Quale ha il silhouette score maggiore?
4. Spiega perché lo scaling è importante in K-Means

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

np.random.seed(42)
X_sin = np.column_stack([
    np.random.randn(200) * 1000,
    np.random.randn(200)
])

X_con = StandardScaler().fit_transform(X_sin)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, X_uso, titulo in zip(axes, [X_sin, X_con], ['Senza scaling', 'Con StandardScaler']):
    km = KMeans(3, n_init=15, random_state=42)
    labels = km.fit_predict(X_uso)
    sc = silhouette_score(X_uso, labels)
    for cl in range(3):
        mask = labels == cl
        ax.scatter(X_uso[mask, 0], X_uso[mask, 1], s=20, alpha=0.7)
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               c='black', marker='X', s=200, zorder=10)
    ax.set_title(f'{titulo}\nSilhouette={sc:.4f}')
    ax.set_xlabel('Feature A'); ax.set_ylabel('Feature B')

plt.suptitle('Importanza dello scaling in K-Means', fontsize=11)
plt.tight_layout(); plt.show()
print("SENZA scaling: la distanza euclidea è dominata dalla feature A (scala 1000x).")
print("K-Means ignora la feature B → i cluster sono bande verticali.")
print("CON scaling: entrambe le feature contribuiscono ugualmente alla distanza.")
```
</details>

---
### Esercizio 3 — Segmentazione di notizie per argomento

**Consegna:** Usa il dataset delle cifre scritte a mano come se fossero dati senza etichetta:

```python
from sklearn.datasets import load_digits
digits = load_digits()
X = digits.data   # 1797 immagini di 64 pixel
y = digits.target # NON usare per allenare
```

1. Riduci a 2D con PCA prima di clusterizzare (per visualizzazione ed efficienza):
   ```python
   from sklearn.decomposition import PCA
   X_2d = PCA(n_components=2, random_state=42).fit_transform(X)
   ```
2. Applica KMeans con K=10 (il numero reale di cifre)
3. Visualizza i cluster in 2D
4. Mostra il silhouette score e commenta il risultato

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X, y = digits.data, digits.target

# Scalare + ridurre la dimensione
X_sc = StandardScaler().fit_transform(X)
X_2d = PCA(n_components=2, random_state=42).fit_transform(X_sc)

# KMeans con K=10
km = KMeans(10, n_init=20, random_state=42)
labels = km.fit_predict(X_2d)
sc = silhouette_score(X_2d, labels)

# Visualizzare
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
scatter = axes[0].scatter(X_2d[:,0], X_2d[:,1], c=labels, cmap='tab10', s=10, alpha=0.7)
axes[0].scatter(km.cluster_centers_[:,0], km.cluster_centers_[:,1],
                c='black', marker='X', s=200, zorder=10)
axes[0].set_title(f'K-Means K=10 (silhouette={sc:.3f})')
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# Etichette reali per confronto
scatter2 = axes[1].scatter(X_2d[:,0], X_2d[:,1], c=y, cmap='tab10', s=10, alpha=0.7)
axes[1].set_title('Classi reali (cifre 0-9)')
plt.colorbar(scatter2, ax=axes[1], label='Cifra')

plt.suptitle('Clustering di cifre in 2D (PCA)', fontsize=11)
plt.tight_layout(); plt.show()

print(f"Silhouette Score con K=10: {sc:.4f}")
print("Commento: silhouette basso perché in 2D le cifre si sovrappongono molto.")
print("Con più componenti PCA (es. 20), il clustering migliorerebbe.")
```
</details>

---
### Esercizio 4 — Sensibilità all'inizializzazione

**Consegna:** Dimostra che K-Means può convergere a minimi locali diversi:

1. Usa `make_blobs(n_samples=200, centers=4, cluster_std=2, random_state=42)`
2. Esegui KMeans 20 volte con `n_init=1` e `init='random'`, seed diversa ogni volta
3. Salva l'inerzia e il silhouette di ogni esecuzione
4. Confronta con un'esecuzione usando `n_init=20` e `init='k-means++'`
5. Perché n_init>1 e k-means++ sono buone pratiche?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X, _ = make_blobs(n_samples=200, centers=4, cluster_std=2, random_state=42)

inercias_random, siluetas_random = [], []
for seed in range(20):
    km = KMeans(4, n_init=1, init='random', random_state=seed)
    lab = km.fit_predict(X)
    inercias_random.append(km.inertia_)
    siluetas_random.append(silhouette_score(X, lab))

# K-Means++ con più tentativi
km_plus = KMeans(4, n_init=20, init='k-means++', random_state=42)
lab_plus = km_plus.fit_predict(X)
sc_plus = silhouette_score(X, lab_plus)

print(f"Inizializzazione random (20 seed):")
print(f"  Inerzia: min={min(inercias_random):.1f}, max={max(inercias_random):.1f}, "
      f"intervallo={max(inercias_random)-min(inercias_random):.1f}")
print(f"  Silhouette: min={min(siluetas_random):.4f}, max={max(siluetas_random):.4f}")
print(f"\nK-Means++ (n_init=20):")
print(f"  Inerzia: {km_plus.inertia_:.1f}")
print(f"  Silhouette: {sc_plus:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(inercias_random, bins=8, color='coral', edgecolor='white')
axes[0].axvline(km_plus.inertia_, color='green', lw=2.5, label='k-means++')
axes[0].set_title('Variabilità dell\'inerzia (init random)'); axes[0].legend()

axes[1].hist(siluetas_random, bins=8, color='steelblue', edgecolor='white')
axes[1].axvline(sc_plus, color='green', lw=2.5, label='k-means++')
axes[1].set_title('Variabilità del silhouette'); axes[1].legend()

plt.tight_layout(); plt.show()
print("\nConclusione: k-means++ seleziona centroidi iniziali ben separati.")
print("n_init>1 garantisce di non restare bloccati in un minimo locale.")
```
</details>

---
### Esercizio 5 (sfida) — Pipeline di segmentazione completa

**Consegna:** Costruisci una funzione `segmentar_clientes(df, features, k_max=10)` che:

1. Scala le feature
2. Determina automaticamente il K ottimale (elbow + silhouette)
3. Allena K-Means con il K scelto
4. Aggiunge una colonna `'segmento'` al DataFrame
5. Restituisce il DataFrame con i segmenti e una tabella riassuntiva per segmento
6. Traccia i segmenti in 2D (usa le prime 2 feature)

Provala con il dataset di clienti di questa lezione.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

def segmentar_clientes(df, features, k_max=10):
    X = df[features].values
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X)

    # Determinare K ottimale
    ks = range(2, k_max+1)
    siluetas = []
    inercias = []
    for k in ks:
        km = KMeans(k, n_init=15, random_state=42)
        lab = km.fit_predict(X_sc)
        siluetas.append(silhouette_score(X_sc, lab))
        inercias.append(km.inertia_)

    k_opt = list(ks)[np.argmax(siluetas)]

    # Allenare il KMeans finale
    km_final = KMeans(k_opt, n_init=20, random_state=42)
    df = df.copy()
    df['segmento'] = km_final.fit_predict(X_sc)

    # Tabella riassuntiva
    resumen = df.groupby('segmento')[features].agg(['mean','count'])

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, (titulo, datos_k, metric) in zip(axes, [
        ('Silhouette', ks, siluetas), ('Inerzia', ks, inercias)
    ]):
        ax.plot(datos_k, metric, 'bo-', ms=7, lw=2)
        ax.axvline(k_opt, color='red', ls='--', label=f'K={k_opt}')
        ax.set_title(titulo); ax.set_xlabel('K'); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    fig2, ax2 = plt.subplots(figsize=(7, 5))
    for seg in range(k_opt):
        mask = df['segmento'] == seg
        ax2.scatter(df.loc[mask, features[0]], df.loc[mask, features[1]],
                    s=25, alpha=0.7, label=f'Segmento {seg}')
    ax2.set_xlabel(features[0]); ax2.set_ylabel(features[1])
    ax2.set_title(f'Segmentazione (K={k_opt})')
    ax2.legend(); plt.tight_layout(); plt.show()

    print(f"K ottimale: {k_opt} | Silhouette: {max(siluetas):.4f}")
    return df, resumen

# Prova con il df_clientes del corpo della lezione
df_cl2 = df_clientes.drop(columns=['segmento','nombre_segmento'], errors='ignore')
features = ['gasto_mensual', 'visitas_mes', 'edad']
df_seg, resumen = segmentar_clientes(df_cl2, features)
print(resumen)
```
</details>

---
## Riepilogo

```python
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# 1. Scalare SEMPRE prima di KMeans
X_sc = StandardScaler().fit_transform(X)

# 2. Scegliere K con Elbow + Silhouette
for k in range(2, 10):
    km = KMeans(k, n_init=15, random_state=42)
    labels = km.fit_predict(X_sc)
    print(k, km.inertia_, silhouette_score(X_sc, labels))

# 3. Allenare il modello finale
km_final = KMeans(n_clusters=4, n_init=20, random_state=42)
labels = km_final.fit_predict(X_sc)

# 4. Attributi utili
km_final.cluster_centers_   # centroidi (nello spazio scalato)
km_final.inertia_            # somma delle distanze² dentro i cluster
km_final.n_iter_             # iterazioni fino a convergenza

# Regola d'oro: SEMPRE scalare → SEMPRE usare n_init>1 → confrontare Elbow + Silhouette
```

## Cosa vediamo nella prossima lezione?
Nella **Lezione 9** impareremo a costruire **Pipeline sklearn** complete e a salvare i modelli con `joblib` per distribuirli in produzione.